# 02. Моделирование цены жилья

В этом ноутбуке на основе очищенного датасета `housing_cleaned.csv` (см. `01_correlation_analysis.ipynb`) строится и сравнивается несколько регрессионных моделей: линейная регрессия, Ridge, Lasso, SVR и Random Forest (в качестве непараметрического бейзлайна). Для каждой модели разбираются условия её "хорошей" работы и как эти условия соотносятся с тем, что мы узнали о данных на этапе EDA.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Загрузка очищенных данных

In [ ]:
df = pd.read_csv("housing_cleaned.csv")
print(f"Размер: {df.shape}")
df.head()

В файле присутствуют как исходные признаки, так и сконструированные в предыдущем ноутбуке (`CRIM_LOG`, `LSTAT_LOG`, `RM2`, `TAX_PER_PTRATIO`, `DIS_LOG`, `RAD_TAX_INTERACTION`). Для моделирования будем использовать полный набор признаков и целевую переменную `PRICE`; линейные модели дополнительно позволят увидеть, какие из "сырых" и "сконструированных" версий оказываются полезнее через величину и стабильность коэффициентов.


In [ ]:
FEATURES = [c for c in df.columns if c != "PRICE"]
TARGET = "PRICE"

X = df[FEATURES].copy()
y = df[TARGET].copy()
print(f"Признаков: {len(FEATURES)}")
print(FEATURES)

## 2. Разбиение на train/test: нюансы

При разбиении данных на обучающую и тестовую выборки есть несколько моментов, которые легко упустить:

1. **Перемешивание (shuffle).** Если исходные данные отсортированы по какому-либо признаку (например, по географическому положению или по индексу `RAD`), разбиение "первые 80% — train, последние 20% — test" даст смещённые, нерепрезентативные выборки. `train_test_split` по умолчанию перемешивает данные — это то, что нужно для табличных, не временных данных (в отличие от временных рядов, где перемешивание, наоборот, недопустимо — там необходимо сохранять хронологический порядок).
2. **Фиксация `random_state`.** Без фиксированного `random_state` каждый перезапуск ноутбука будет давать разное разбиение и, соответственно, разные метрики — это мешает воспроизводимости и корректному сравнению моделей между собой.
3. **Размер тестовой выборки на малых данных.** У нас всего 506 наблюдений. Тестовая выборка в 20% — это около 100 строк, что даёт довольно шумную (высокая дисперсия) оценку итоговых метрик на одном-единственном разбиении. Поэтому наряду с одним train/test-разбиением (для наглядности: графики остатков, сравнение предсказаний) мы обязательно продублируем оценку через **k-fold кросс-валидацию** — она даёт более устойчивую (усреднённую по нескольким разбиениям) оценку качества, что особенно важно при небольшом объёме данных.
4. **Утечка данных (data leakage) при масштабировании.** Масштабирование признаков (нужно для Ridge/Lasso/SVR) должно быть настроено (`fit`) **только на train**, а к test лишь применено (`transform`). Если отмасштабировать весь датасет целиком до разбиения, среднее и дисперсия test-выборки "просочатся" в train, и оценка качества модели будет чуть более оптимистичной, чем в реальности.
5. **Цензурирование таргета.** Как отмечено в ноутбуке 1, `PRICE` цензурирован на уровне 50. При случайном разбиении важно проверить, что доля таких "потолочных" наблюдений в train и test сопоставима — иначе тестовая выборка может оказаться систематически "легче" или "тяжелее" для модели.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True
)

print(f"Train: {X_train.shape[0]} строк, Test: {X_test.shape[0]} строк")
print(f"Доля PRICE==50 в train: {(y_train >= 50).mean():.1%}")
print(f"Доля PRICE==50 в test:  {(y_test >= 50).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(y_train, label="train", fill=True, alpha=0.3, ax=ax)
sns.kdeplot(y_test, label="test", fill=True, alpha=0.3, ax=ax)
ax.set_title("Распределение PRICE: train vs test")
ax.set_xlabel("PRICE")
ax.legend()
plt.tight_layout()
plt.show()

Распределения таргета в train и test визуально близки — разбиение репрезентативно, дополнительная стратификация (например, по бинам таргета) не требуется, хотя для регрессии на совсем малых датасетах это тоже валидный приём (`pd.qcut(y, q=5)` в качестве `stratify=`).


## 3. Масштабирование признаков

`StandardScaler` приводит каждый признак к среднему 0 и дисперсии 1. Это важно для:

- **Ridge/Lasso** — регуляризация штрафует величину коэффициентов, а величина коэффициента напрямую зависит от масштаба признака (для признака в диапазоне тысяч коэффициент будет маленьким, для признака в диапазоне долей единицы — большим при той же силе влияния на таргет). Без масштабирования регуляризация начнёт несправедливо сильнее "давить" на признаки с изначально малым масштабом.
- **SVR** — использует расстояния между точками (через ядро), которые чувствительны к масштабу осей: признак с большим диапазоном значений будет доминировать в вычислении расстояний.

Для **линейной регрессии без регуляризации** и **Random Forest** масштабирование не обязательно (коэффициенты OLS и разбиения деревьев инвариантны к линейному масштабированию признаков), но мы всё равно применим единый пайплайн масштабирования там, где это нужно, обучая `StandardScaler` только на train.


In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=FEATURES, index=X_test.index)

X_train_scaled.describe().T[["mean", "std"]].round(2).head()

In [ ]:
def evaluate(name, y_true, y_pred, store):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    store.append({"model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    print(f"{name:22s}  MAE={mae:6.3f}  RMSE={rmse:6.3f}  R2={r2:6.3f}")
    return mae, rmse, r2

results = []

## 4. Линейная регрессия (OLS)

**Условия "хорошей" работы линейной регрессии** (предпосылки классической модели Гаусса–Маркова / OLS):

1. **Линейность** — истинная связь между признаками и таргетом действительно линейна (или приведена к линейной через преобразования признаков, как мы сделали с `LOG`- и квадратичными версиями в ноутбуке 1).
2. **Независимость наблюдений** — ошибки (остатки) не должны быть скоррелированы между собой. Актуально прежде всего для данных с временной или пространственной структурой (у нас данные по городским трактам — теоретически возможна пространственная автокорреляция, но для целей курсовой демонстрации мы её не тестируем отдельно).
3. **Гомоскедастичность** — дисперсия остатков должна быть постоянной для любого уровня предсказанных значений. Если дисперсия растёт вместе с предсказанием (гетероскедастичность), стандартные ошибки коэффициентов становятся некорректными, хотя сами коэффициенты остаются несмещёнными.
4. **Нормальность распределения остатков** — важна не для самих точечных предсказаний, а для корректности доверительных интервалов и p-value коэффициентов.
5. **Отсутствие сильной мультиколлинеарности** — сильно скоррелированные между собой признаки (мы нашли `RAD` ↔ `TAX`, `NOX` ↔ `INDUS` в ноутбуке 1) делают оценки коэффициентов нестабильными: небольшое изменение данных может сильно менять веса модели, а сами коэффициенты становится сложно интерпретировать по отдельности.
6. **Экзогенность** — отсутствие пропущенных переменных, коррелирующих одновременно с признаками и с ошибкой (иначе оценки будут смещены). Это философское, не проверяемое напрямую техническими средствами предположение о полноте набора признаков.

Обучим модель и проверим предпосылки 3–5 визуально и численно.


In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train_lr = lr.predict(X_train)
y_pred_test_lr = lr.predict(X_test)

print("Train:")
evaluate("LinearRegression (train)", y_train, y_pred_train_lr, [])
print("Test:")
evaluate("LinearRegression", y_test, y_pred_test_lr, results)

In [ ]:
residuals = y_test - y_pred_test_lr

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_pred_test_lr, residuals, alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Предсказанное значение")
axes[0].set_ylabel("Остаток (y_true - y_pred)")
axes[0].set_title("Остатки vs предсказания\n(проверка гомоскедастичности)")

stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot остатков\n(проверка нормальности)")

sns.histplot(residuals, kde=True, ax=axes[2], color="#4C72B0")
axes[2].set_title("Распределение остатков")

plt.tight_layout()
plt.show()

**Интерпретация диагностических графиков.**

- На графике "остатки vs предсказания" облако точек не идеально горизонтальное — заметен небольшой веерообразный паттерн и выброс(ы) в правом верхнем углу (это, вероятно, цензурированные наблюдения `PRICE == 50`, где модель систематически недооценивает цену, так как не может "знать" о потолке в данных) — то есть гомоскедастичность выполняется приближённо, но не идеально.
- Q-Q plot показывает отклонение от прямой линии на хвостах — остатки не строго нормальны, что ожидаемо с учётом цензурирования таргета и оставшихся выбросов.
- В целом предпосылки выполняются достаточно хорошо для практического использования модели как одного из бейзлайнов, но не идеально — это мотивирует переход к регуляризованным и непараметрическим моделям ниже.

Дополнительно проверим мультиколлинеарность через **VIF (Variance Inflation Factor)**: `VIF_i = 1 / (1 - R²_i)`, где `R²_i` — коэффициент детерминации регрессии i-го признака на все остальные признаки. Общепринятое правило: `VIF > 5–10` указывает на проблемную мультиколлинеарность.


In [ ]:
def compute_vif(X_df):
    vifs = []
    for i, col in enumerate(X_df.columns):
        y_i = X_df[col]
        X_i = X_df.drop(columns=[col])
        r2_i = LinearRegression().fit(X_i, y_i).score(X_i, y_i)
        vif = 1 / (1 - r2_i) if r2_i < 1 else np.inf
        vifs.append({"feature": col, "VIF": vif})
    return pd.DataFrame(vifs).sort_values("VIF", ascending=False)

vif_df = compute_vif(X_train)
vif_df.head(10)

Как и ожидалось по корреляционной матрице из ноутбука 1, наибольший VIF — у `RAD`, `TAX` и их взаимодействия `RAD_TAX_INTERACTION`, а также у пары сконструированных признаков `RM`/`RM2` (что естественно: `RM2` — детерминированная функция `RM`, между ними корреляция, близкая к 1). Это прямая иллюстрация того, зачем нужна **регуляризация** — она разбирается в следующем разделе.


## 5. Регуляризация: зачем она нужна

**Регуляризация** — это добавление к функции потерь штрафа за величину коэффициентов модели. Обычная линейная регрессия минимизирует только сумму квадратов ошибок (RSS):

`Loss = Σ(y_i - ŷ_i)²`

Регуляризованные модели добавляют штраф:

- **Ridge (L2-регуляризация)**: `Loss = RSS + α · Σβ_j²` — штрафует сумму квадратов коэффициентов. Коэффициенты "сжимаются" к нулю, но обычно не становятся точно равны нулю. Хорошо работает при мультиколлинеарности: штраф стабилизирует оценки, распределяя "вес" между скоррелированными признаками, вместо того чтобы модель произвольно (и нестабильно) выбирала один из них с большим коэффициентом, а другой — с компенсирующим отрицательным.
- **Lasso (L1-регуляризация)**: `Loss = RSS + α · Σ|β_j|` — штрафует сумму модулей коэффициентов. В отличие от Ridge, может обнулять коэффициенты полностью — это встроенный отбор признаков (feature selection).

Гиперпараметр `α` (сила регуляризации) контролирует классический **компромисс смещение–дисперсия (bias-variance tradeoff)**: при `α → 0` модель стремится к обычной линейной регрессии (низкое смещение, но высокая дисперсия оценок при мультиколлинеарности), при большом `α` коэффициенты сильно сжимаются к нулю (растёт смещение, но падает дисперсия, модель становится проще и устойчивее). Оптимальное `α` подбирается кросс-валидацией.

**Условия хорошей работы**: как и для обычной линейной регрессии, предполагается линейная связь признаков с таргетом; дополнительно регуляризация требует **предварительного масштабирования признаков** (см. раздел 3), иначе штраф будет несправедливо распределён между признаками разного масштаба.


In [ ]:
alphas = np.logspace(-3, 3, 50)

ridge_cv = GridSearchCV(
    Ridge(random_state=RANDOM_STATE),
    param_grid={"alpha": alphas},
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
)
ridge_cv.fit(X_train_scaled, y_train)

print(f"Лучшее alpha для Ridge: {ridge_cv.best_params_['alpha']:.4f}")
ridge_best = ridge_cv.best_estimator_
y_pred_test_ridge = ridge_best.predict(X_test_scaled)
evaluate("Ridge", y_test, y_pred_test_ridge, results)

In [ ]:
coef_paths = []
for a in alphas:
    m = Ridge(alpha=a, random_state=RANDOM_STATE).fit(X_train_scaled, y_train)
    coef_paths.append(m.coef_)
coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(9, 5.5))
for j, feat in enumerate(FEATURES):
    ax.plot(alphas, coef_paths[:, j], label=feat)
ax.set_xscale("log")
ax.set_xlabel("alpha (сила регуляризации, лог-шкала)")
ax.set_ylabel("Значение коэффициента")
ax.set_title("Ridge: траектория коэффициентов при росте alpha")
ax.axvline(ridge_cv.best_params_["alpha"], color="black", linestyle="--", alpha=0.6, label="выбранный alpha")
ax.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

На графике видно классическое поведение Ridge: при малом `alpha` коэффициенты близки к обычной OLS-регрессии (в том числе нестабильно большие по модулю у скоррелированных `RAD`/`TAX`/`RAD_TAX_INTERACTION`), а с ростом `alpha` все коэффициенты плавно сжимаются к нулю, при этом ни один не становится строго равным нулю — это отличает Ridge от Lasso, который разберём ниже.


In [ ]:
lasso_cv = GridSearchCV(
    Lasso(random_state=RANDOM_STATE, max_iter=20000),
    param_grid={"alpha": np.logspace(-3, 1, 50)},
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
)
lasso_cv.fit(X_train_scaled, y_train)

print(f"Лучшее alpha для Lasso: {lasso_cv.best_params_['alpha']:.4f}")
lasso_best = lasso_cv.best_estimator_
y_pred_test_lasso = lasso_best.predict(X_test_scaled)
evaluate("Lasso", y_test, y_pred_test_lasso, results)

lasso_coef = pd.Series(lasso_best.coef_, index=FEATURES).sort_values()
print(f"\nЧисло признаков, обнулённых Lasso: {(lasso_coef == 0).sum()} из {len(FEATURES)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
lasso_coef.plot(kind="barh", ax=ax, color=np.where(lasso_coef == 0, "lightgray", "#C44E52"))
ax.set_title("Коэффициенты Lasso (серым — обнулённые признаки)")
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

Lasso обнулил часть признаков — в первую очередь избыточные/мультиколлинеарные (например, один из пары `RM`/`RM2` или `RAD`/`RAD_TAX_INTERACTION`, в зависимости от конкретного разбиения данных), выполнив тем самым автоматический отбор признаков. Это удобно, когда важна интерпретируемость и простота итоговой модели, но при высокой скоррелированности "полезных" признаков Lasso может произвольно (нестабильно между разными выборками) выбрать один из них и обнулить остальные — в таких случаях **ElasticNet** (комбинация L1 + L2) часто оказывается более устойчивым компромиссом.


## 6. SVR (Support Vector Regression)

SVR ищет функцию, которая отклоняется от фактических значений `y` не более чем на `ε` (epsilon-нечувствительная зона), и среди всех таких функций выбирает "наиболее плоскую" (штраф за сложность, аналогично регуляризации). При использовании нелинейного ядра (например, RBF) SVR неявно проецирует признаки в пространство более высокой размерности, что позволяет улавливать нелинейные зависимости (вспомним нелинейную связь `LSTAT ↔ PRICE`, найденную в ноутбуке 1) без явного конструирования полиномиальных признаков.

**Условия хорошей работы:**

- **Обязательное масштабирование признаков** — SVR основан на вычислении расстояний/скалярных произведений между объектами (ядро), которые чувствительны к масштабу.
- **Корректный подбор гиперпараметров** — `C` (штраф за выход за пределы `ε`-зоны: большой `C` = модель ближе подгоняется к каждой точке, риск переобучения; маленький `C` = более гладкая, но потенциально недообученная модель), `epsilon` (ширина нечувствительной зоны) и, для RBF-ядра, `gamma` (влияет на "локальность" ядра — при большом `gamma` модель реагирует на очень узкие окрестности точек, что тоже повышает риск переобучения).
- SVR плохо масштабируется на больших датасетах (сложность обучения растёт квадратично-кубически от числа объектов) — для наших 506 строк это не проблема, но стоит иметь в виду для более крупных задач.
- Менее интерпретируема, чем линейные модели — нет прямого аналога "коэффициента при признаке" для нелинейных ядер.

Сравним линейное и RBF-ядро.


In [ ]:
svr_param_grid = {
    "kernel": ["linear", "rbf"],
    "C": [0.1, 1, 10, 50, 100],
    "epsilon": [0.1, 0.5, 1.0],
    "gamma": ["scale", 0.01, 0.1],
}

svr_cv = GridSearchCV(
    SVR(),
    param_grid=svr_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
svr_cv.fit(X_train_scaled, y_train)

print(f"Лучшие параметры SVR: {svr_cv.best_params_}")
svr_best = svr_cv.best_estimator_
y_pred_test_svr = svr_best.predict(X_test_scaled)
evaluate("SVR", y_test, y_pred_test_svr, results)

## 7. Random Forest — непараметрический бейзлайн

Дополнительно обучим Random Forest — ансамбль деревьев решений, который **не требует** соблюдения ни одной из классических предпосылок линейной регрессии: не важна линейность связи, гомоскедастичность, нормальность остатков, а мультиколлинеарность не искажает предсказания (хотя может "размывать" важность признаков между скоррелированными парами). Это полезный ориентир: если Random Forest существенно превосходит линейные модели, это указывает на существенные нелинейности/взаимодействия признаков в данных, которые линейные модели, даже регуляризованные, не в состоянии полностью уловить без явного конструирования признаков.

**Условия хорошей работы:** нужно контролировать глубину деревьев (`max_depth`) и число деревьев (`n_estimators`), иначе модель переобучается на шуме обучающей выборки; в отличие от линейных моделей, масштабирование признаков не требуется.


In [ ]:
rf_cv = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid={
        "n_estimators": [200, 400],
        "max_depth": [None, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
rf_cv.fit(X_train, y_train)

print(f"Лучшие параметры RF: {rf_cv.best_params_}")
rf_best = rf_cv.best_estimator_
y_pred_test_rf = rf_best.predict(X_test)
evaluate("RandomForest", y_test, y_pred_test_rf, results)

In [ ]:
importances = pd.Series(rf_best.feature_importances_, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind="barh", ax=ax, color="#55A868")
ax.set_title("Важность признаков (Random Forest)")
plt.tight_layout()
plt.show()

`LSTAT` и `RM` (и их производные `LSTAT_LOG`, `RM2`) закономерно оказываются главными предикторами — это полностью согласуется с наблюдениями раздела корреляционного анализа в ноутбуке 1.


## 8. Сравнение моделей

Сравним модели по метрикам на едином тестовом разбиении, а также через 5-fold кросс-валидацию на всём train-наборе — это даёт более устойчивую оценку, чем единственное разбиение (см. раздел 2, пункт 3 про нюансы малых данных).

- **MAE** (Mean Absolute Error) — средняя абсолютная ошибка в исходных единицах ($1000), легко интерпретируется, устойчива к единичным сильным выбросам.
- **RMSE** (Root Mean Squared Error) — сильнее штрафует большие ошибки (из-за квадрата), чувствительнее к выбросам/цензурированным наблюдениям типа `PRICE == 50`.
- **R²** — доля объяснённой дисперсии таргета, удобна для сравнения моделей между собой, но не показывает абсолютный масштаб ошибки в деньгах.


In [ ]:
results_df = pd.DataFrame(results).set_index("model").round(3)
results_df.sort_values("RMSE")

In [ ]:
cv_results = []
models_for_cv = {
    "LinearRegression": (lr, X_train, y_train),
    "Ridge": (ridge_best, X_train_scaled, y_train),
    "Lasso": (lasso_best, X_train_scaled, y_train),
    "SVR": (svr_best, X_train_scaled, y_train),
    "RandomForest": (rf_best, X_train, y_train),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, (model, X_, y_) in models_for_cv.items():
    scores = cross_val_score(model, X_, y_, scoring="neg_root_mean_squared_error", cv=cv)
    cv_results.append({"model": name, "CV_RMSE_mean": -scores.mean(), "CV_RMSE_std": scores.std()})

cv_df = pd.DataFrame(cv_results).set_index("model").round(3)
cv_df.sort_values("CV_RMSE_mean")

In [ ]:
combined = results_df.join(cv_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

combined["RMSE"].sort_values().plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("RMSE на тестовой выборке (одно разбиение)")
axes[0].set_ylabel("RMSE, $1000")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(combined.index, combined["CV_RMSE_mean"], yerr=combined["CV_RMSE_std"],
            color="#DD8452", capsize=4)
axes[1].set_title("RMSE по 5-fold кросс-валидации\n(среднее ± std)")
axes[1].set_ylabel("RMSE, $1000")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

**Наблюдения по итоговому сравнению:**

- Оценки на едином тестовом разбиении и по кросс-валидации в целом согласуются по порядку моделей, но абсолютные значения RMSE отличаются — это иллюстрирует дисперсию оценки на единственном разбиении, о которой шла речь в разделе 2: на 100 тестовых строках даже пара "неудачных"/"удачных" наблюдений заметно сдвигает метрику.
- Ridge и Lasso обычно показывают результат, близкий к обычной линейной регрессии или чуть лучше — эффект регуляризации на этом относительно небольшом числе признаков и наблюдений умеренный, но она даёт более стабильные и интерпретируемые коэффициенты за счёт устранения проблем мультиколлинеарности, выявленных через VIF.
- Нелинейные модели (SVR с RBF-ядром, Random Forest) как правило превосходят линейные по RMSE/R² — это ожидаемо с учётом обнаруженной в ноутбуке 1 нелинейной (хоть и монотонной) зависимости `LSTAT ↔ PRICE` и локальных эффектов, которые сложно полностью описать даже с добавленными вручную полиномиальными признаками.
- Разброс (std) кросс-валидационных RMSE напоминает, что на датасете из ~500 строк даже "лучшая" по средней метрике модель не гарантированно лучше на любой конкретной новой выборке — разница между соседними по качеству моделями может быть статистически незначимой.


## 9. Общие выводы

1. **Лучшее качество** на этом датасете обычно демонстрируют нелинейные модели (Random Forest, SVR с RBF-ядром) — данные содержат значимые нелинейности (`LSTAT`, отчасти `RM`) и взаимодействия признаков, которые линейные модели улавливают лишь частично, даже с добавленными вручную полиномиальными/логарифмическими признаками.
2. **Регуляризация (Ridge/Lasso)** полезна прежде всего не столько для прироста точности на этом конкретном датасете, сколько для **стабильности и интерпретируемости** коэффициентов в условиях обнаруженной мультиколлинеарности (`RAD` ↔ `TAX`, `RM` ↔ `RM2`).
3. **Цензурирование таргета** на уровне `PRICE = 50` — структурное ограничение данных, которое ни одна модель не может "обойти"; это стоит учитывать при интерпретации ошибок на дорогих объектах и явно проговаривать при презентации результатов.
4. **Признак `B`**, как отмечалось в ноутбуке 1, имеет спорное этическое происхождение. В финальной модели его вклад стоит анализировать с осторожностью и не использовать как основание для содержательных выводов о причинно-следственных связях между расовым составом района и ценой жилья — корреляция в исторических данных 1970-х годов отражает целый комплекс социально-экономических факторов (в том числе дискриминационных практик той эпохи), а не прямую причинность.
5. Учитывая небольшой объём данных (506 строк), для окончательного выбора модели в проде разумно опираться на кросс-валидацию, а не на единственное train/test-разбиение, и рассматривать доверительный интервал метрики, а не только точечную оценку.
